# Soft X-Ray Signal Analysis

This notebook demonstrates the VEST SXR workflow using the `soft_x_rays` OMAS IDS and `vaft.plot` helpers. The sequence follows the usual diagnostic reading order: line-of-sight geometry, representative time trace, spectrogram, and chord-time pattern.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from vaft.machine_mapping.soft_x_rays import soft_x_rays_from_digitizer_csv
from vaft.plot import (
    plot_soft_x_ray_los,
    plot_soft_x_ray_signal,
    plot_soft_x_ray_spectrogram,
    plot_soft_x_ray_pattern,
    plot_soft_x_ray_overview,
)

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 200

## 1. Build a Soft X-Ray ODS

The example below prefers the VEST SXR tutorial raw data if it is present, and otherwise falls back to the packaged `vaft` sample. The resulting ODS contains both brightness traces and static line-of-sight metadata.

In [ ]:
shot = 45538
daq_label = 22577

vest_raw = Path("/Users/yun/git/VEST_Soft X-ray/data/raw")
plasma_file = vest_raw / f"digitizer_{daq_label}_{shot}.csv"

if plasma_file.exists():
    ods = soft_x_rays_from_digitizer_csv(shot, daq_label, digitizer_file=plasma_file)
else:
    shot = 45531
    ods = soft_x_rays_from_digitizer_csv(shot, daq_label)

print(f"shot={shot}, daq_label={daq_label}")
print(f"channels={len(ods['soft_x_rays.channel'])}, samples={len(ods['soft_x_rays.time'])}")
print(ods['soft_x_rays.ids_properties.source'])

## 2. Line-of-Sight Geometry

This plot uses `soft_x_rays.channel[:].line_of_sight` and separates the array groups by color. The packaged geometry table carries the toroidal angle `phi`, so the plotted section can be tied back to the 12 o’clock / 4 o’clock SXR locations.

In [ ]:
plot_soft_x_ray_los(
    ods,
    arrays=["lowermid", "bottom"],
    show_channel_labels=True,
    title="SXR LOS Geometry: two-filter arrays",
);

## 3. Representative Signal

The time trace is read from `brightness.data`. These digitizer values are stored as a brightness proxy unless a calibrated `brightness_scale` is supplied during mapping.

In [ ]:
time_range = (0.304, 0.328)
baseline_range = (0.304, 0.306)
representative_channels = [0, 16]  # lowermid ch 1 and bottom ch 1 for DAQ 22577

plot_soft_x_ray_signal(
    ods,
    channels=representative_channels,
    time_range=time_range,
    baseline_range=baseline_range,
    ylabel="SXR brightness proxy [a.u.]",
    title="Representative SXR traces",
);

## 4. Spectrogram

The spectrogram helper infers the sampling rate from the ODS time base. Use a shorter `nperseg` for better time localization near rapidly evolving internal MHD activity.

In [ ]:
plot_soft_x_ray_spectrogram(
    ods,
    channel=16,
    time_range=time_range,
    baseline_range=baseline_range,
    nperseg=512,
    noverlap=384,
    max_frequency=90_000,
    title="Bottom SXR ch 1 spectrogram",
);

## 5. SXR Chord-Time Pattern

This heatmap arranges a channel block as chord number versus time. It is useful for seeing radial/poloidal pattern motion before moving to filtered mode analysis.

In [ ]:
bottom_channels = list(range(16, 32))

plot_soft_x_ray_pattern(
    ods,
    channels=bottom_channels,
    time_range=time_range,
    baseline_range=baseline_range,
    orientation="time_vertical",
    title="Bottom-array SXR chord-time pattern",
);

## 6. Compact Overview

The same helpers can be combined into a compact four-panel overview for quick shot screening.

In [ ]:
plot_soft_x_ray_overview(
    ods,
    los_arrays=["lowermid", "bottom"],
    signal_channels=representative_channels,
    spectrogram_channel=16,
    pattern_channels=bottom_channels,
    time_range=time_range,
    baseline_range=baseline_range,
);